# Lesson 22: SQL & Data Wrangling for ML

*ML Academy — companion notebook*

Pull training data out of databases: SELECT, GROUP BY, joins, window functions, pandas equivalents and point-in-time features.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-22.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 3. SELECT, WHERE and ORDER BY

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")                 # a database that lives in memory
pd.read_csv("../data/houses.csv").to_sql("houses", con, index=False)
viewings = pd.DataFrame({"house_id": [1, 1, 2, 4, 4, 4, 5, 8, 11, 11, 12],
                         "day":      [3, 9, 2, 1, 4, 8, 6, 5, 2, 7, 1]})
viewings.to_sql("viewings", con, index=False)

q = """SELECT id, area, size_m2, price_lakh
       FROM houses
       WHERE price_lakh > 80
       ORDER BY price_lakh DESC"""
print(pd.read_sql(q, con))

   id       area  size_m2  price_lakh
0  12     Suburb      125          96
1   3    Central      120          95
2   5     Suburb      110          88
3   8  Outskirts      150          84
4   2    Central       90          82


## 4. GROUP BY: from rows to summaries

In [2]:
q = """SELECT area, COUNT(*) AS homes, ROUND(AVG(price_lakh), 1) AS avg_price,
              ROUND(AVG(price_lakh * 1.0 / size_m2), 3) AS lakh_per_m2
       FROM houses
       GROUP BY area
       HAVING COUNT(*) >= 3
       ORDER BY avg_price DESC"""
print(pd.read_sql(q, con))

        area  homes  avg_price  lakh_per_m2
0    Central      4       76.8        0.889
1     Suburb      5       76.4        0.692
2  Outskirts      3       58.7        0.462


## 5. Joins: combining tables

In [3]:
inner = """SELECT h.id, COUNT(v.day) AS n_viewings
           FROM houses h JOIN viewings v ON v.house_id = h.id
           GROUP BY h.id"""
left = """SELECT h.id, COUNT(v.day) AS n_viewings
          FROM houses h LEFT JOIN viewings v ON v.house_id = h.id
          GROUP BY h.id"""
print("inner join rows:", len(pd.read_sql(inner, con)), "  left join rows:", len(pd.read_sql(left, con)))
print(pd.read_sql(left, con).T)

inner join rows: 7   left join rows: 12
            0   1   2   3   4   5   6   7   8   9   10  11
id           1   2   3   4   5   6   7   8   9  10  11  12
n_viewings   2   1   0   3   1   0   0   1   0   0   2   1


## 7. Window functions: features from context

In [4]:
q = """SELECT id, area, price_lakh,
              RANK() OVER (PARTITION BY area ORDER BY price_lakh DESC) AS rank_in_area,
              ROUND(price_lakh - AVG(price_lakh) OVER (PARTITION BY area), 1) AS vs_area_avg
       FROM houses
       ORDER BY area, rank_in_area"""
print(pd.read_sql(q, con))

    id       area  price_lakh  rank_in_area  vs_area_avg
0    3    Central          95             1         18.3
1    2    Central          82             2          5.3
2   11    Central          72             3         -4.8
3    1    Central          58             4        -18.8
4    8  Outskirts          84             1         25.3
5   10  Outskirts          50             2         -8.7
6    9  Outskirts          42             3        -16.7
7   12     Suburb          96             1         19.6
8    5     Suburb          88             2         11.6
9    6     Suburb          79             3          2.6
10   7     Suburb          64             4        -12.4
11   4     Suburb          55             5        -21.4


## 8. The same operations in pandas

In [5]:
houses = pd.read_sql("SELECT * FROM houses", con)
n_view = viewings.groupby("house_id").size().rename("n_viewings")
merged = houses.merge(n_view, left_on="id", right_index=True, how="left").fillna({"n_viewings": 0})
summary = merged.groupby("area").agg(homes=("id", "size"), avg_price=("price_lakh", "mean"),
                                     avg_viewings=("n_viewings", "mean")).round(2)
print(summary)

           homes  avg_price  avg_viewings
area                                     
Central        4      76.75          1.25
Outskirts      3      58.67          0.33
Suburb         5      76.40          1.00


## 10. Point-in-time correctness: the rule that prevents leakage

In [6]:
# Point-in-time features: count only viewings that happened BEFORE the prediction day
predict_day = 5
q = """SELECT h.id, COUNT(v.day) AS viewings_so_far
       FROM houses h LEFT JOIN viewings v
            ON v.house_id = h.id AND v.day < ?
       GROUP BY h.id"""
past = pd.read_sql(q, con, params=(predict_day,))
allv = pd.read_sql(left, con)
print("viewings counted, all time:   ", int(allv.n_viewings.sum()))
print("viewings counted, before day 5:", int(past.viewings_so_far.sum()))

viewings counted, all time:    11
viewings counted, before day 5: 6


## 11. In Python: from database to training table

In [7]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
pd.read_csv("../data/houses.csv").to_sql("houses", con, index=False)
pd.DataFrame({"house_id": [1, 1, 2, 4, 4, 4, 5, 8, 11, 11, 12],
              "day": [3, 9, 2, 1, 4, 8, 6, 5, 2, 7, 1]}).to_sql("viewings", con, index=False)

query = """
WITH v AS (                                  -- step 1: summarise events before the cut-off
    SELECT house_id, COUNT(*) AS n_viewings, MIN(day) AS first_view
    FROM viewings WHERE day < 5 GROUP BY house_id
)
SELECT h.*,                                  -- step 2: one row per house, every house kept
       COALESCE(v.n_viewings, 0) AS n_viewings,
       AVG(h.price_lakh) OVER (PARTITION BY h.area) AS area_avg_price
FROM houses h LEFT JOIN v ON v.house_id = h.id
"""
train = pd.read_sql(query, con)
print(train.shape)
print(train[["id", "area", "n_viewings", "area_avg_price", "sold_fast"]].head())

(12, 11)
   id       area  n_viewings  area_avg_price  sold_fast
0   1    Central           1       76.750000          1
1   2    Central           1       76.750000          1
2   3    Central           0       76.750000          0
3  11    Central           1       76.750000          1
4   8  Outskirts           0       58.666667          0


## 14. Going deeper: indexes, NULL logic and rolling features

In [8]:
import sqlite3
import numpy as np
import pandas as pd

# A bigger events table: 200,000 viewings of 5,000 houses
rng = np.random.default_rng(0)
con = sqlite3.connect(":memory:")
pd.DataFrame({"house_id": rng.integers(1, 5001, 200000), "day": rng.integers(1, 366, 200000)}).to_sql("viewings", con, index=False)
q = "SELECT COUNT(*) FROM viewings WHERE house_id = 42"
print("plan without an index:", con.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][-1])
con.execute("CREATE INDEX idx_house ON viewings(house_id)")
print("plan with an index:   ", con.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][-1])
print("viewings of house 42:", con.execute(q).fetchone()[0])

plan without an index: SCAN viewings


plan with an index:    SEARCH viewings USING COVERING INDEX idx_house (house_id=?)
viewings of house 42: 37


In [9]:
# NULL is "unknown": comparisons with it are neither true nor false
t = pd.DataFrame({"id": [1, 2, 3, 4], "discount": [10, None, 0, None]})
t.to_sql("t", con, index=False)
conds = {"discount > 5": "discount > 5", "NOT (discount > 5)": "NOT (discount > 5)",
         "discount equals NULL": "discount = NULL", "discount IS NULL": "discount IS NULL"}
for label, cond in conds.items():
    print(f"{label:21} -> rows {[r[0] for r in con.execute(f'SELECT id FROM t WHERE {cond}')]}")
print("COUNT(*), COUNT(discount), AVG(discount):", con.execute("SELECT COUNT(*), COUNT(discount), AVG(discount) FROM t").fetchone())

discount > 5          -> rows [1]
NOT (discount > 5)    -> rows [3]
discount equals NULL  -> rows []
discount IS NULL      -> rows [2, 4]
COUNT(*), COUNT(discount), AVG(discount): (4, 2, 5.0)


In [10]:
# Rolling, point-in-time features with a window frame: viewings in the 7 days up to and including each day
daily = pd.DataFrame({"house_id": [1] * 10, "day": range(1, 11), "views": [0, 2, 1, 0, 3, 1, 0, 0, 4, 1]})
daily.to_sql("daily", con, index=False)
q = """SELECT day, views,
              SUM(views) OVER (PARTITION BY house_id ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS views_7d,
              LAG(views) OVER (PARTITION BY house_id ORDER BY day) AS views_yesterday
       FROM daily"""
print(pd.read_sql(q, con).to_string(index=False))

 day  views  views_7d  views_yesterday
   1      0         0              NaN
   2      2         2              0.0
   3      1         3              2.0
   4      0         3              1.0
   5      3         6              0.0
   6      1         7              3.0
   7      0         7              1.0
   8      0         7              0.0
   9      4         9              0.0
  10      1         9              4.0


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [11]:
import sqlite3
import pandas as pd
df = pd.read_csv("../data/houses.csv")
con = sqlite3.connect(":memory:")
df.to_sql("houses", con, index=False)
q = """SELECT area, COUNT(*) AS homes, SUM(sold_fast) AS sold_fast,
              ROUND(AVG(sold_fast), 2) AS fast_rate, ROUND(AVG(price_lakh), 1) AS avg_price
       FROM houses GROUP BY area ORDER BY fast_rate DESC"""
print(pd.read_sql(q, con))

        area  homes  sold_fast  fast_rate  avg_price
0    Central      4          3       0.75       76.8
1     Suburb      5          3       0.60       76.4
2  Outskirts      3          0       0.00       58.7


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [12]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Average price per area with SQL

Complete the query so it returns the average price of each area.

In [13]:
import sqlite3
import pandas as pd
con = sqlite3.connect(":memory:")
pd.DataFrame({"area": ["A", "A", "B", "B", "B"], "price": [10, 20, 30, 40, 50]}).to_sql("t", con, index=False)
q = "SELECT area, price AS avg_price FROM t"  # ← replace with a GROUP BY query
result = pd.read_sql(q, con)["avg_price"].round(2).tolist()
check(result, [15.0, 40.0], hint='SELECT the area and AVG(price) AS avg_price, then GROUP BY area (and ORDER BY area).')

❌ Not yet. Hint: SELECT the area and AVG(price) AS avg_price, then GROUP BY area (and ORDER BY area).


### Challenge 2: A left join in pandas

Merge the counts onto every customer, keeping customers with no orders, and fill their count with 0.

In [14]:
import pandas as pd
customers = pd.DataFrame({"id": [1, 2, 3, 4]})
orders = pd.DataFrame({"customer_id": [1, 1, 3]})
counts = orders.groupby("customer_id").size().rename("n").reset_index()
merged = customers.merge(counts, left_on="id", right_on="customer_id")  # ← keep every customer
result = merged["n"].tolist()
check(result, [2.0, 0.0, 1.0, 0.0], hint='Use how="left" and fill missing counts with 0.')

❌ Not yet. Hint: Use how="left" and fill missing counts with 0.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Average price per area with SQL** — the missing line:

```python
q = "SELECT area, AVG(price) AS avg_price FROM t GROUP BY area ORDER BY area"
```

**2. A left join in pandas** — the missing line:

```python
merged = customers.merge(counts, left_on="id", right_on="customer_id", how="left").fillna({"n": 0})
```